In [ ]:
import ollama
import os
import json
from my_tools import TOOLS, AVAILABLE_FUNCTIONS

In [ ]:
def chat_with_tools(user_message: str, model: str = "granite4.1:8b"):
    messages = [{'role': 'user', 'content': user_message}]

    response = ollama.chat(
        model=model,
        messages=messages,
        tools=TOOLS
    )

    messages.append(response['message'])

    if response['message'].get('tool_calls'):
        print("Model is calling tools...\n")
        
        for tool_call in response['message']['tool_calls']:
            function_name = tool_call['function']['name']
            function_args = tool_call['function']['arguments']
            
            print(f"Calling function: {function_name}")
            print(f"Arguments: {json.dumps(function_args, indent=2)}\n")
            
            if function_name in AVAILABLE_FUNCTIONS:
                function_to_call = AVAILABLE_FUNCTIONS[function_name]
                function_response = function_to_call(**function_args)
                
                print(f"Function response: {function_response}\n")
                
                messages.append({
                    'role': 'tool',
                    'content': function_response,
                })
        
        final_response = ollama.chat(model=model, messages=messages)
        assistant_message = final_response['message']['content']
    else:
        assistant_message = response['message']['content']

    print("="*80)
    print(f"Assistant: {assistant_message}")
    print("="*80)
    return assistant_message

user_message = "please send mail to example@example.com with subject 'Hello' and message 'Hello, how are you?"


In [ ]:
chat_with_tools(user_message)